In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.de_pro COMMENT 'DE Professional Exam practice';
CREATE VOLUME IF NOT EXISTS workspace.de_pro.raw COMMENT 'raw landing files';


In [0]:
%sql
SHOW TABLES IN samples.tpch;

In [0]:
import json, random, uuid, datetime as dt

BASE = "/Volumes/workspace/de_pro/raw"
for d in ["orders", "customers_cdc", "events_json"]:
    dbutils.fs.mkdirs(f"{BASE}/{d}")

COUNTRIES = ["IN", "US", "UK", "DE", "SG"]

def write_orders(n=500, bad_ratio=0.05):
    now = dt.datetime.utcnow()
    rows = []
    for _ in range(n):
        amt = round(random.uniform(50, 5000), 2)
        if random.random() < bad_ratio:
            amt = -1  # bad record for data-quality demos
        rows.append({
            "order_id": str(uuid.uuid4()),
            "customer_id": random.randint(1, 200),
            "country": random.choice(COUNTRIES),
            "amount": amt,
            "order_ts": (now - dt.timedelta(minutes=random.randint(0, 120))).isoformat()
        })
    path = f"{BASE}/orders/orders_{now:%Y%m%d%H%M%S}.json"
    dbutils.fs.put(path, "\n".join(json.dumps(r) for r in rows), overwrite=True)
    print("wrote", path)

_seq = {"v": 0}
def write_customer_cdc(n=50):
    rows = []
    for _ in range(n):
        _seq["v"] += 1
        rows.append({
            "customer_id": random.randint(1, 60),
            "name": f"Customer {random.randint(1, 999)}",
            "email": f"user{random.randint(1, 999)}@example.com",
            "city": random.choice(["Bengaluru", "Mumbai", "London", "Austin"]),
            "operation": random.choices(["INSERT", "UPDATE", "DELETE"], [6, 3, 1])[0],
            "seq": _seq["v"]
        })
    path = f"{BASE}/customers_cdc/cdc_{dt.datetime.utcnow():%Y%m%d%H%M%S}.json"
    dbutils.fs.put(path, "\n".join(json.dumps(r) for r in rows), overwrite=True)
    print("wrote", path)

def write_events(n=200):
    rows = []
    for _ in range(n):
        rows.append({
            "event_id": str(uuid.uuid4()),
            "payload": {"device": random.choice(["ios", "android", "web"]),
                        "geo": {"country": random.choice(COUNTRIES)},
                        "items": [{"sku": f"SKU{random.randint(1, 50)}", "qty": random.randint(1, 4)}]},
            "review": random.choice(["Loved it, fast delivery", "Terrible packaging", "Okay product", "Will buy again!"])
        })
    path = f"{BASE}/events_json/events_{dt.datetime.utcnow():%Y%m%d%H%M%S}.json"
    dbutils.fs.put(path, "\n".join(json.dumps(r) for r in rows), overwrite=True)
    print("wrote", path)

write_orders(); write_customer_cdc(); write_events()
display(dbutils.fs.ls(f"{BASE}/orders"))